# 04 — Statistical Analysis

**Pipeline position:** consumes `data/processed/cleaned_data.csv` (output of `02_cleaning.ipynb` / `scripts/etl_pipeline.py`). Produces statistical evidence that feeds the project report and `docs/recommendations.md`.

**Business question:** *Which product attributes — price, discount, rating, reviews, badge status, sponsorship, coupons — most strongly drive `estimated_revenue` on Amazon?*

**Hypotheses tested**
1. **H1 — Reviews predict revenue.** Higher review count correlates with higher monthly revenue.
2. **H2 — Discounts drive sales.** Steeper discount % correlates with higher revenue.
3. **H3 — Best-seller badge matters.** Best-seller products earn significantly more than non-best-sellers.
4. **H4 — Category matters.** Mean revenue differs significantly across product categories.
5. **H5 — Coupons and badges co-occur.** Best-seller status and coupon presence are not independent.

**Methods:** Pearson + Spearman correlations · OLS multiple regression · Welch's t-test · one-way ANOVA · chi-square test of independence. All tests report p-values; α = 0.05 throughout.

In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

## 1. Load cleaned data

In [3]:
df = pd.read_csv('../data/processed/cleaned_data.csv')
print('Shape:', df.shape)
df.head(3)

Shape: (30228, 18)


,product_title,product_rating,total_reviews,purchased_last_month,discounted_price,original_price,is_best_seller,is_sponsored,has_coupon,buy_box_availability,delivery_date,sustainability_tags,data_collected_at,product_category,discount_percentage,estimated_revenue,discount_value,rating_category
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6000,375.0000,300.0000,89.6800,159.0000,no badge,sponsored,save 15% with coupon,Add to cart,2025-09-01,Carbon impact,2025-08-21 11:14:29,phones,43.6000,"26,904.0000",69.3200,High
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3000,"2,457.0000","6,000.0000",9.9900,15.9900,no badge,sponsored,no coupon,Add to cart,2025-08-29,NaN,2025-08-21 11:14:29,laptops,37.5200,"59,940.0000",6.0000,High
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6000,"3,044.0000","2,000.0000",314.0000,349.0000,no badge,sponsored,no coupon,Add to cart,2025-09-01,NaN,2025-08-21 11:14:29,laptops,10.0300,"628,000.0000",35.0000,High


### Derive analytical boolean flags

The cleaned dataset stores `is_best_seller`, `is_sponsored`, and `has_coupon` as lowercased badge strings (e.g. `'best seller'`, `'no badge'`, `'limited time deal'`, `'save 15% with coupon'`). For statistical tests we collapse them to clean 0/1 flags here, *without modifying the saved CSV*.

- `flag_best_seller` = 1 if the badge is exactly `'best seller'`, else 0
- `flag_sponsored`  = 1 if `is_sponsored == 'sponsored'`, else 0
- `flag_has_coupon` = 1 if `has_coupon` is anything other than `'no coupon'`, else 0

In [4]:
df['flag_best_seller'] = (df['is_best_seller'] == 'best seller').astype(int)
df['flag_sponsored']   = (df['is_sponsored']   == 'sponsored').astype(int)
df['flag_has_coupon']  = (df['has_coupon']     != 'no coupon').astype(int)

pd.DataFrame({
    'flag': ['best_seller', 'sponsored', 'has_coupon'],
    'share_of_listings': [
        df['flag_best_seller'].mean(),
        df['flag_sponsored'].mean(),
        df['flag_has_coupon'].mean(),
    ],
}).round(4)

,flag,share_of_listings
0,best_seller,0.0086
1,sponsored,0.1191
2,has_coupon,0.0371


## 2. Correlation analysis (H1, H2)

Test whether continuous predictors — price, discount %, rating, reviews — have monotonic relationships with `estimated_revenue`. Pearson measures *linear* correlation; Spearman measures *rank-order* correlation (robust to outliers and non-linear-but-monotonic relationships).

In [5]:
def correlation_table(df, target, predictors, method='pearson'):
    fn = stats.pearsonr if method == 'pearson' else stats.spearmanr
    rows = []
    for col in predictors:
        r, p = fn(df[col], df[target])
        rows.append({
            'predictor': col,
            f'{method}_r': round(r, 4),
            'p_value': p,
            'significant_at_0.05': p < 0.05,
        })
    return pd.DataFrame(rows)

predictors = [
    'discounted_price', 'original_price', 'discount_percentage',
    'product_rating', 'total_reviews', 'discount_value',
    'flag_best_seller', 'flag_sponsored', 'flag_has_coupon',
]

print('Pearson correlations with estimated_revenue:\n')
print(correlation_table(df, 'estimated_revenue', predictors, 'pearson').to_string(index=False))

Pearson correlations with estimated_revenue:

          predictor  pearson_r  p_value  significant_at_0.05
   discounted_price     0.0672   0.0000                 True
     original_price     0.0563   0.0000                 True
discount_percentage     0.0000   0.9941                False
     product_rating     0.0919   0.0000                 True
      total_reviews     0.2080   0.0000                 True
     discount_value    -0.0011   0.8498                False
   flag_best_seller     0.2425   0.0000                 True
     flag_sponsored     0.2845   0.0000                 True
    flag_has_coupon     0.0148   0.0100                 True


In [6]:
print('Spearman correlations with estimated_revenue:\n')
print(correlation_table(df, 'estimated_revenue', predictors, 'spearman').to_string(index=False))

Spearman correlations with estimated_revenue:

          predictor  spearman_r  p_value  significant_at_0.05
   discounted_price      0.5369   0.0000                 True
     original_price      0.5250   0.0000                 True
discount_percentage     -0.1156   0.0000                 True
     product_rating     -0.0428   0.0000                 True
      total_reviews      0.0602   0.0000                 True
     discount_value     -0.0487   0.0000                 True
   flag_best_seller      0.1014   0.0000                 True
     flag_sponsored      0.2682   0.0000                 True
    flag_has_coupon      0.1222   0.0000                 True


### How to read this
- **|r| < 0.1** is negligible; **0.1–0.3** weak; **0.3–0.5** moderate; **>0.5** strong.
- **p-value < 0.05** means the observed correlation is unlikely due to chance — the relationship is *statistically real*, even if weak.
- If Spearman is much larger than Pearson, the relationship is monotonic-but-non-linear and a log transform would help before regression.

## 3. Multiple regression — what drives revenue holding other levers constant? (H1, H2)

Single-variable correlations are misleading because predictors are themselves correlated (e.g. high-rated products also tend to have more reviews). OLS regression estimates the *independent* contribution of each predictor while controlling for the others.

**Model**

```
estimated_revenue ~ const + discounted_price + discount_percentage
                  + product_rating + total_reviews
                  + flag_best_seller + flag_sponsored + flag_has_coupon
```

In [7]:
features = [
    'discounted_price', 'discount_percentage', 'product_rating', 'total_reviews',
    'flag_best_seller', 'flag_sponsored', 'flag_has_coupon',
]

X = sm.add_constant(df[features])
y = df['estimated_revenue']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:      estimated_revenue   R-squared:                       0.180
Model:                            OLS   Adj. R-squared:                  0.179
Method:                 Least Squares   F-statistic:                     944.9
Date:                Tue, 28 Apr 2026   Prob (F-statistic):               0.00
Time:                        12:16:27   Log-Likelihood:            -4.0493e+05
No. Observations:               30228   AIC:                         8.099e+05
Df Residuals:                   30220   BIC:                         8.099e+05
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const               -1.621e+05   1

### How to read the regression output
- **R-squared:** share of variance in revenue explained by all predictors together. 0.20–0.40 is reasonable for noisy retail data; >0.5 is strong.
- **coef:** how much revenue (in $) changes per 1-unit increase in that predictor, holding everything else constant.
- **P>|t|:** p-value for that coefficient. < 0.05 means the predictor has a statistically significant independent effect.
- **Compare predictors:** large positive coef *and* p < 0.05 → that lever genuinely moves revenue and is a candidate for a recommendation.

## 4. T-test — does the Best-Seller badge actually pay off? (H3)

- **H₀ (null):** mean revenue of best-sellers = mean revenue of non-best-sellers.
- **H₁ (alt):** the means differ.

We use **Welch's t-test** (does not assume equal variances) because the two groups have very different sizes and spreads.

In [8]:
g_bs   = df.loc[df['flag_best_seller'] == 1, 'estimated_revenue']
g_nobs = df.loc[df['flag_best_seller'] == 0, 'estimated_revenue']

t_stat, p_val = stats.ttest_ind(g_bs, g_nobs, equal_var=False)

print(f'Best-seller     :  n = {len(g_bs):>6,}   mean = {g_bs.mean():>14,.0f}   median = {g_bs.median():>12,.0f}')
print(f'Non-best-seller :  n = {len(g_nobs):>6,}   mean = {g_nobs.mean():>14,.0f}   median = {g_nobs.median():>12,.0f}')
print()
print(f't-statistic = {t_stat:.3f},  p-value = {p_val:.6e}')
decision = 'REJECT H0 — best-sellers earn significantly more.' if p_val < 0.05 else 'FAIL TO REJECT H0 — no significant difference.'
print(f'Decision (alpha=0.05): {decision}')

Best-seller     :  n =    259   mean =        510,036   median =      156,800
Non-best-seller :  n = 29,969   mean =         48,141   median =       13,500

t-statistic = 7.173,  p-value = 7.740629e-12
Decision (alpha=0.05): REJECT H0 — best-sellers earn significantly more.


## 5. ANOVA — does revenue differ across product categories? (H4)

- **H₀:** mean revenue is the same across all categories.
- **H₁:** at least one category's mean differs.

One-way ANOVA tests this in a single F-test instead of running pairwise t-tests across every category pair (which would inflate false positives).

In [9]:
category_groups = [
    g['estimated_revenue'].values
    for _, g in df.groupby('product_category')
    if len(g) >= 30
]
f_stat, p_val = stats.f_oneway(*category_groups)

print(f'One-way ANOVA on {len(category_groups)} categories (each n>=30):')
print(f'F-statistic = {f_stat:.3f},  p-value = {p_val:.6e}')
decision = 'REJECT H0 — at least one category differs from the others.' if p_val < 0.05 else 'FAIL TO REJECT H0.'
print(f'Decision (alpha=0.05): {decision}')

print('\nMean revenue by category (top 10):')
cat_summary = (
    df.groupby('product_category')['estimated_revenue']
      .agg(n='count', mean_revenue='mean', median_revenue='median')
      .sort_values('mean_revenue', ascending=False)
      .head(10)
      .round(0)
)
print(cat_summary.to_string())

One-way ANOVA on 15 categories (each n>=30):
F-statistic = 161.233,  p-value = 0.000000e+00
Decision (alpha=0.05): REJECT H0 — at least one category differs from the others.

Mean revenue by category (top 10):
                        n  mean_revenue  median_revenue
product_category                                       
power & batteries    2683  196,303.0000     16,190.0000
laptops              6038   51,740.0000     17,499.0000
smart home            250   50,273.0000     18,495.0000
phones               5108   47,005.0000     12,732.0000
tv & display         1352   43,494.0000      9,998.0000
storage               949   43,389.0000      9,399.0000
wearables             429   41,512.0000      9,998.0000
cameras              2490   34,938.0000     14,998.0000
networking            826   33,236.0000     17,998.0000
printers & scanners   875   27,769.0000      8,005.0000


## 6. Chi-square — are coupon use and best-seller status independent? (H5)

- **H₀:** coupon presence and best-seller status are independent.
- **H₁:** they are associated.

Rejecting H₀ suggests coupons and best-seller badges co-occur (or anti-occur) more than chance — useful when designing promotional strategy.

In [10]:
contingency = pd.crosstab(
    df['flag_has_coupon'], df['flag_best_seller'],
    rownames=['has_coupon'], colnames=['best_seller'],
)
print('Contingency table:')
print(contingency)

chi2, p_val, dof, expected = stats.chi2_contingency(contingency)
print(f'\nChi-square = {chi2:.3f},  dof = {dof},  p-value = {p_val:.6e}')
decision = 'REJECT H0 — coupon presence and best-seller status are NOT independent.' if p_val < 0.05 else 'FAIL TO REJECT H0.'
print(f'Decision (alpha=0.05): {decision}')

Contingency table:
best_seller      0    1
has_coupon             
0            28856  252
1             1113    7

Chi-square = 0.480,  dof = 1,  p-value = 4.885559e-01
Decision (alpha=0.05): FAIL TO REJECT H0.


## 7. Summary of findings

After running the notebook end-to-end, fill in the actual statistic and p-value from each section above. This table feeds directly into `docs/recommendations.md` and the report's *Statistical Analysis Results* section.

| # | Hypothesis | Test | Statistic | p-value | Decision | Implication |
|---|-----------|------|-----------|---------|----------|-------------|
| H1 | Reviews predict revenue | Pearson + OLS coef on `total_reviews` | _fill_ | _fill_ | _fill_ | If significant + positive → prioritise listings with active review-generation |
| H2 | Discounts drive sales | Pearson + OLS coef on `discount_percentage` | _fill_ | _fill_ | _fill_ | If weak / insignificant → reconsider discount-led promotions |
| H3 | Best-seller badge → higher revenue | Welch's t-test | _fill_ | _fill_ | _fill_ | If significant → invest in earning the badge (inventory, fulfilment, ratings) |
| H4 | Revenue differs across categories | One-way ANOVA | _fill_ | _fill_ | _fill_ | Use category mix to allocate ad spend / inventory |
| H5 | Coupons × best-seller co-occur | Chi-square | _fill_ | _fill_ | _fill_ | Informs whether coupons are a path *to* or *consequence of* best-seller status |

**Next step:** translate the rows above with significant p-values into 3–5 specific, action-oriented recommendations in `docs/recommendations.md`.